In [1]:
import joblib

X_train, X_test, y_train, y_test = joblib.load(
    "ds52-train_test_split.joblib"
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (36168, 15)
X_test: (9043, 15)
y_train: (36168,)
y_test: (9043,)


## Feature Engineering

New features are created using information available before making the marketing decision. The same transformations are applied independently to the training and test datasets. The target variable is not used during feature creation, preventing target leakage.

1.1 import libraries

In [2]:
import pandas as pd
import numpy as np


1.2 protects member 1 original dataset.

In [3]:
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

1.3 The `pdays` value of `-1` means that the customer was not contacted in a previous campaign. A binary feature is created to represent this information clearly.

In [5]:
X_train_fe["previously_contacted"] = (
    X_train_fe["pdays"] != -1
).astype(int)

In [6]:
X_test_fe["previously_contacted"] = (
    X_test_fe["pdays"] != -1
).astype(int)

In [7]:
print(X_train_fe["previously_contacted"].value_counts())

previously_contacted
0    29584
1     6584
Name: count, dtype: int64


In [8]:
X_train_fe[["pdays", "previously_contacted"]].head()

,pdays,previously_contacted
24001,-1,0
43409,185,1
20669,-1,0
18810,-1,0
23130,-1,0


In [10]:
X_train_fe.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,previously_contacted
24001,36,technician,divorced,secondary,no,861,no,no,telephone,29,aug,2,-1,0,unknown,0
43409,24,student,single,secondary,no,4126,no,no,cellular,5,apr,4,185,7,failure,1
20669,44,technician,single,secondary,no,244,yes,no,cellular,12,aug,4,-1,0,unknown,0
18810,48,unemployed,married,secondary,no,0,no,no,telephone,31,jul,11,-1,0,unknown,0
23130,38,technician,married,secondary,no,257,no,no,cellular,26,aug,10,-1,0,unknown,0


1.4 Final Obligation Count.

This feature counts whether the customer has a housing loan, personal loan, or credit default. A higher value may indicate greater financial commitments.

In [11]:
X_train_fe["financial_obligation_count"] = (
    (X_train_fe["housing"] == "yes").astype(int)
    + (X_train_fe["loan"] == "yes").astype(int)
    + (X_train_fe["default"] == "yes").astype(int)
)

In [12]:
X_test_fe["financial_obligation_count"] = (
    (X_test_fe["housing"] == "yes").astype(int)
    + (X_test_fe["loan"] == "yes").astype(int)
    + (X_test_fe["default"] == "yes").astype(int)
)

In [13]:
print(
    X_train_fe["financial_obligation_count"]
    .value_counts()
    .sort_index()
)

financial_obligation_count
0    13540
1    18725
2     3792
3      111
Name: count, dtype: int64


| Count | Meaning                                          | Customers |
| ----: | ------------------------------------------------ | --------: |
|     0 | No housing loan, personal loan or credit default |    13,540 |
|     1 | One of the three conditions                      |    18,725 |
|     2 | Two conditions                                   |     3,792 |
|     3 | All three conditions                             |       111 |


1.5 Previous Campaign Success


Creating a new feature.This binary feature indicates whether the customer responded successfully to a previous marketing campaign.

In [14]:
X_train_fe["previous_campaign_success"] = (
    X_train_fe["poutcome"] == "success"
).astype(int)

In [15]:
X_test_fe["previous_campaign_success"] = (
    X_test_fe["poutcome"] == "success"
).astype(int)

In [16]:
print(X_train_fe["previous_campaign_success"].value_counts())

previous_campaign_success
0    34963
1     1205
Name: count, dtype: int64


1.6 Customer Age Group

Customers are placed into age groups to capture possible non-linear differences in campaign response between younger, middle-aged, and older customers.

In [17]:
age_bins = [17, 30, 40, 50, 60, np.inf]

In [18]:


age_labels = [
    "18-30",
    "31-40",
    "41-50",
    "51-60",
    "60+"
]

In [19]:
X_train_fe["age_group"] = pd.cut(
    X_train_fe["age"],
    bins=age_bins,
    labels=age_labels
)

In [20]:
X_test_fe["age_group"] = pd.cut(
    X_test_fe["age"],
    bins=age_bins,
    labels=age_labels
)

In [21]:

print(X_train_fe["age_group"].value_counts().sort_index())

age_group
18-30     5703
31-40    14135
41-50     8983
51-60     6378
60+        969
Name: count, dtype: int64


In [22]:
X_train_fe[["age", "age_group"]].head(10)

,age,age_group
24001,36,31-40
43409,24,18-30
20669,44,41-50
18810,48,41-50
23130,38,31-40
15058,48,41-50
15908,50,41-50
30424,46,41-50
9998,46,41-50
14935,45,41-50
